# Create zarr stroring unstitched tiles

Ome-zarr libs:
https://github.com/ome/ome-zarr-py/issues/407

In [ ]:
import glob
from ngff_zarr import to_ngff_image, to_multiscales, to_ngff_zarr
import tifffile
import zarr

## Source data

In [ ]:
source_path = 'C:/Project/slides/example_cells/*.tiff'

filenames = glob.glob(source_path)
filenames

## Store as zarr

In [ ]:
pyramid_downsample = 2
npyramid_add = 2
ome_version = '0.5'
pixel_size = {'x': 0.004, 'y': 0.004, 'z': 1}


def read_tiff(filename):
    tif = tifffile.TiffFile(filename)
    data = tif.asarray()
    metadata = {}
    return data, metadata


def write_tile(path, data, dim_order, pixel_size, translation=None):
    axes_units = {dim: 'micrometer' for dim in dim_order if dim in 'xyz'}
    image = to_ngff_image(data, dims=dim_order, scale=pixel_size,
                          translation=translation, axes_units=axes_units)

    scale_factors = [pyramid_downsample ** (scale + 1) for scale in range(npyramid_add)]
    multiscales = to_multiscales(image, scale_factors=scale_factors, chunks=1024)

    chunks_per_shard = None

    to_ngff_zarr(path, multiscales, chunks_per_shard=chunks_per_shard, version=ome_version)

    return multiscales.metadata

## Store as zarr

In [ ]:
for filename in filenames:
    data, metadata = read_tiff(filename)
    print(filename, data.shape, data.dtype)
    tile_name = filename.split('/')[-1].split('.')[0]
    path = f'output/{tile_name}.zarr'
    write_tile(path, data, dim_order='zyx', pixel_size=pixel_size)